# 01 — Korpus & Annotation

Phase 1 (Korpus + Bias-Notiz) und Phase 2 (κ + drei Edge Cases) leben in diesem Notebook. Pipeline → `02_extract.ipynb`, Eval → `03_eval.ipynb`, Frontier → `04_frontier_compare.ipynb`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum (Phase 1) | 2026-05-04 & 2026-05-11 |
| Datum (Phase 2) | 2026-05-11 |
| Korpus-Datei | `daten/eigener_korpus.jsonl` |
| Anzahl Anzeigen im Korpus | 44 |
| Genutzte Suchanfragen (Phase 1) | Data Analyst, Data Engineer, Business Intelligence, Fachinformatiker Daten- und Prozessanalyse, Data Scientist -> jeweils in Bremen |
| Pair-Partner:in (Phase 2) | Fabio Bostanci |
| 12 gemeinsame Anzeigen-IDs | _ |

## Phase 1 — Korpus inspizieren + Bias-Notiz

**Beschaffen der Daten über API Request und generieren des Korpus**

In [4]:
import requests
import json
import time
import base64
from pathlib import Path

API_KEY = "jobboerse-jobsuche"
BASE_URL = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4"

HEADERS = {
    "X-API-Key": API_KEY
}

suchanfragen = [
    {"was": "Data Analyst", "wo": "Bremen"},
    {"was": "Data Engineer", "wo": "Bremen"},
    {"was": "Business Intelligence", "wo": "Bremen"},
    {"was": "Fachinformatiker Daten- und Prozessanalyse", "wo": "Bremen"},
    {"was": "Data Scientist", "wo": "Bremen"},
]

anzeigen = {}

for suche in suchanfragen:
    print("Suche:", suche)

    response = requests.get(
        f"{BASE_URL}/jobs",
        headers=HEADERS,
        params={
            "was": suche["was"],
            "wo": suche["wo"],
            "size": 20,
        }
    )

    print("Status:", response.status_code)

    daten = response.json() # dictonary aus Antwort erstellen
    treffer = daten.get("stellenangebote", []) # Liste mit Stellenangeboten ziehen
    
# Detailsuche für gefundene Treffer anhand Refnr um an stellenangebotsBeschreibung zu kommen
    for treffer_item in treffer:
        refnr = treffer_item.get("refnr")

        if not refnr:
            continue

        if refnr in anzeigen: # falls Anzeige bereits mit anderem Suchbegriff gefunden wurde und doppelt auftaucht
            continue

        refnr_encoded = base64.b64encode(refnr.encode("utf-8")).decode("utf-8") # damit das Anhängen an die URL keine Fehler durch Sonderzeichen wirft

        detail_response = requests.get(
            f"{BASE_URL}/jobdetails/{refnr_encoded}",
            headers=HEADERS
        )

        if detail_response.status_code != 200:
            print("Detail fehlgeschlagen:", refnr, detail_response.status_code)
            continue

        detail = detail_response.json()

        anzeige = {
            "refnr": refnr,
            "titel": detail.get("stellenangebotsTitel"),
            "firma": detail.get("firma"),
            "text": detail.get("stellenangebotsBeschreibung"),
            "ort": detail.get("stellenlokationen", [{}])[0].get("adresse", {}).get("ort"),
            "beruf": detail.get("hauptberuf"),
            "veroeffentlichung": detail.get("datumErsteVeroeffentlichung"),
            "homeoffice_api": detail.get("homeofficemoeglich"),
            "gehalt_api": detail.get("verguetungsangabe"),
            "vertragsdauer_api": detail.get("vertragsdauer"),
            "raw": detail # Originalantwort der API
        }

        if anzeige["text"]: #anzeige nur im dic speichern, wenn tatsächlich stellenbeschreibung vorhanden
            anzeigen[refnr] = anzeige

        time.sleep(0.5) # Pause zwischen Abfragen

print("Gesammelte Anzeigen:", len(anzeigen))

Suche: {'was': 'Data Analyst', 'wo': 'Bremen'}
Status: 200
Suche: {'was': 'Data Engineer', 'wo': 'Bremen'}
Status: 200
Suche: {'was': 'Business Intelligence', 'wo': 'Bremen'}
Status: 200
Suche: {'was': 'Fachinformatiker Daten- und Prozessanalyse', 'wo': 'Bremen'}
Status: 200
Suche: {'was': 'Data Scientist', 'wo': 'Bremen'}
Status: 200
Gesammelte Anzeigen: 44


In [5]:
output_path = Path("../daten/eigener_korpus.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f: #w -> überschreiben
    for anzeige in anzeigen.values():
        f.write(json.dumps(anzeige, ensure_ascii=False) + "\n") # ensure_ascii=False -> Umlaute erlauben; + "\n" danach neue Zeile

print("Gespeichert:", output_path)
print("Anzahl gespeicherter Anzeigen:", len(anzeigen))

Gespeichert: ../daten/eigener_korpus.jsonl
Anzahl gespeicherter Anzeigen: 44


In [7]:
with open("../daten/eigener_korpus.jsonl", "r", encoding="utf-8") as f:
    zeilen = f.readlines()

print("Zeilen in Datei:", len(zeilen))

erste_anzeige = json.loads(zeilen[0])
erste_anzeige.keys()

Zeilen in Datei: 44


dict_keys(['refnr', 'titel', 'firma', 'text', 'ort', 'beruf', 'veroeffentlichung', 'homeoffice_api', 'gehalt_api', 'vertragsdauer_api', 'raw'])

**Korpus Exploration**

In [2]:
import pandas as pd

korpus = pd.read_json("../daten/eigener_korpus.jsonl", lines=True)
korpus.head()

<jemalloc>: Unsupported system page size


,refnr,titel,firma,text,ort,beruf,veroeffentlichung,homeoffice_api,gehalt_api,vertragsdauer_api,raw
0,10001-1002993453-S,Data Scientist/Analyst (m/w/d),SThree Temp Experts GmbH,## **Deine Aufgaben**\n\n- Du analysierst und ...,Bremen,Data-Analyst/in,2026-04-28,NaN,JAHRESGEHALT,BEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
1,10001-1002928664-S,Data Analyst (m/w/d),Wolters Rundreisen GmbH Sachbearbeiter/in,**STARTE MIT UNS DEINE NEUE REISE – WIR FREUEN...,Stuhr,Business-Analyst/in,2026-04-16,1.0,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
2,10001-1003016654-S,Data Analyst (m/w/d) Schwerpunkt BI,Orange Engineering GmbH & Co. KG,Im Auftrag unseres namhaften Kunden aus dem Ra...,Bremen,Data-Analyst/in,2026-05-04,NaN,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
3,10001-1002870609-S,Data Analyst / Data Engineer (m/w/d),Team Business IT GmbH,"## Aufgaben, die deine Neugier wecken:\n\nDu h...",Rostock,Data Engineer,2026-04-02,1.0,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
4,10001-1003011936-S,Data Analyst (m/w/d) für AIRBUS,SimpleXX GmbH,**Data Analyst (m/w/d) für AIRBUS**\n\n\_\_\_\...,Bremen,Bachelor Professional - IT (Datenanalyse),2026-05-02,NaN,KEINE_ANGABEN,KEINE_ANGABE,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."


Erste Auffälligkeiten:
- bei Ort taucht Stuhr und Rostock auf obwohl in Bremen gesucht werden sollte
- Index 4 hat im Titel ...für AIRBUS stehen und bei Firma SimpleXXGmbH (Könnte sich um ein Headhunter Unternehmen handeln) -> möglicher Kandidat für EdgeCase, da mehrere Interpretationen möglich
- einige Texte scheinen mit Markdown-Sonderzeichen durchzogen zu sein

In [24]:
korpus["text"].str.len().describe()

count      44.000000
mean     2848.772727
std      1093.728695
min       899.000000
25%      2217.750000
50%      2698.000000
75%      3258.750000
max      5678.000000
Name: text, dtype: float64

Hohe Standardabweichung bei Textlänge der Stellenbeschreibung
-> Texte unterscheiden sich sich stark in Länge und bewegen sich zwischen 899 und 5678 Zeichenlänge

In [26]:
# number of unique values per column
korpus.drop(columns=["raw"]).nunique(dropna=False)

refnr                44
titel                43
firma                29
text                 43
ort                   9
beruf                24
veroeffentlichung    26
homeoffice_api        3
gehalt_api            3
vertragsdauer_api     3
dtype: int64

In [17]:
cols_toCheck = ['titel', 'firma', 'ort', 'beruf', 'veroeffentlichung', 'homeoffice_api', 'gehalt_api', 'vertragsdauer_api']
for col in cols_toCheck:
    print(f"\nSpalte: {col}")
    print(korpus[col].value_counts(dropna=False))



Spalte: titel
titel
Electrical Engineer (m/w/d)                                                                           2
Data Scientist/Analyst (m/w/d)                                                                        1
Controller (m/w/d)                                                                                    1
Requirements Engineer (m/w/d)                                                                         1
Senior QA Engineer (m/w/d) – Testautomatisierung & OT-Softwarequalität - ab sofort - VZ               1
Technical Sales Engineer Mitte (Mittel- Südhessen bis Saarland) (w/m/d) - ab sofort - VZ              1
Software Engineer DevOps (m/w/d)                                                                      1
Geschäftsfeldsteuerer - Signal and Data Intelligence (m/w/d)                                          1
Duales Studium Bachelor of Science Angewandte Künstliche Intelligenz (w/d/m) - 2027                   1
Sachbearbeitung (m/w/d) Projektabrechnung /

Auffälligkeiten:
- __Jobtitel__ unterscheiden sich von der Syntax -> häufig Jobtitel + (m/w/d), aber auch genauere Spezifikationen, Jahreszahl, Vollzeit, Firma; Erfahrungslevel als Nennungen vorhanden; sogar 1x nur "**  Ausbildung 2026 **"; fast keine Mehrfachnennungen; einige Jobtitel wirken nicht so ganz passend z.B. Electrical Engineer und Sachbearbeitung (m/w/d) Projektabrechnung / Faktura 
- __Firma__ hat einige Mehrfachnennungen; auf ersten Blick keine Nennung von gleicher Firma mit unterschiedlicher Schreibweise; Rheinmetal 8x könnte Bias erzeugen
- __Ort__ hat noch mehr Orte neben Stuhr und Rostock, die nicht Bremen sind (insgesamt 7) -> Ursachenforschung (möglicherweise full remote?)
- __Veröffentlichung__ zeigt, dass die meisten Anzeigen aus den letzten Monaten stammen, vereinzelt auch 2025 vertreten
- __Homeoffice__ mit 0,1 oder NaN angegeben als float
- __Gehalt__ nur in 9 Fällen überhaupt Angaben
- __Vertragsdauer__ besser gepflegt, aber trotzdem noch 16 mal keine Angabe

-> drauf achten NaN oder KEINE_ANGABE
-> Senioritätslevel in diesen Daten nur vereinzelt im Jobtitel angegeben, sonst nirgendwo


In [34]:
# show homeoffice_api for ort != Bremen
korpus.loc[korpus["ort"] != "Bremen", ["ort", "homeoffice_api"]]

,ort,homeoffice_api
1,Stuhr,1.0
3,Rostock,1.0
8,"Aurich, Ostfriesland",0.0
16,Lemwerder,NaN
18,Lemwerder,NaN
19,Kaiserslautern,1.0
31,Oldenburg (Oldb),1.0
37,Oyten,NaN
43,Hamburg,0.0


-> nicht nur Remotestellen

In [37]:
pattern = r"(\*\*|#{1,6}\s|\n{3,}|_{3,}|-{3,}|\*{3,})" # schaut nach **, #, ##, ###, viele aufeinanderfolgende Zeilenumbrüche, Trennlinie aus Unterstrichen oder Sternchen

korpus["text"].str.contains(pattern, regex=True, na=False).value_counts()

/tmp/ipykernel_101/1636005936.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  korpus["text"].str.contains(pattern, regex=True, na=False).value_counts()


text
True     35
False     9
Name: count, dtype: int64

In [38]:
pattern_2 = r"(\*\*|#{1,6})"

korpus["text"].str.contains(pattern_2, regex=True, na=False).value_counts()

/tmp/ipykernel_101/2923112929.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  korpus["text"].str.contains(pattern_2, regex=True, na=False).value_counts()


text
True     26
False    18
Name: count, dtype: int64

In [39]:
for i, row in korpus.head(3).iterrows():
    print(f"RefNr: {row['refnr']}")
    print(f"Titel: {row['titel']}")
    print(f"Firma: {row['firma']}")
    print(row["text"][:3000])  # begrenzt Länge
    print("\n" + "="*100 + "\n") 

RefNr: 10001-1002993453-S
Titel: Data Scientist/Analyst (m/w/d)
Firma: SThree Temp Experts GmbH
## **Deine Aufgaben**

- Du analysierst und strukturierst **technische und logistische Daten** aus Konstruktion, Betrieb und Instandhaltung
- Du führst **logistische Systemanalysen** durch und erstellst aussagekräftige Berichte im Rahmen des **Integrated Logistics Support (ILS)**
- Du bewertest **Zuverlässigkeit, Wartungskonzepte, Ersatzteilstrategien** sowie **Lebenszykluskosten (Life Cycle Costing)**
- Du identifizierst Optimierungspotenziale in **Wartungs‑ und Logistikprozessen** mithilfe datengetriebener Methoden
- Du entwickelst und pflegst **Datenmodelle sowie Analyse‑ und Auswertungstools**
- Du arbeitest eng mit der **Konstruktion und dem Projektteam** über den gesamten Systemlebenszyklus hinweg zusammen

Für einen Kunden aus dem Großraum Berlin suche ich aktuell nach folgendem Profil:

## **Dein Profil**

- Abgeschlossenes **technisches Studium**, z. B. Ingenieurwissenschaften, Data

In [42]:
sample = korpus.sample(3, random_state=42)

for i, row in sample.iterrows():
    print(f"RefNr: {row['refnr']}")
    print(f"Titel: {row['titel']}")
    print(f"Firma: {row['firma']}")
    print(row["text"][:3000])
    print("\n" + "="*100 + "\n")

RefNr: 10000-1203695504-S
Titel: Ausbildung Fachinformatiker/in – Fachrichtung Daten- und Prozessanalyse 2026 m/w/d
Firma: Kreyenhop & Kluge GmbH & Co. KG
Kreyenhop & Kluge ist ein traditionsreicher, weltweit aktiver Lebensmittelimporteur. Wir versorgen Industrie und Handel mit Spezialitäten aus aller Welt, insbesondere aus dem asiatischen Raum.

Wir suchen zum Ausbildungsbeginn am 01.08.2026, oder 01.09.2026 Auszubildende für den Beruf der/des Fachinformatiker/in – Fachrichtung Daten- und Prozessanalyse.

Dein Profil:

- mindestens ein mittlerer Schulabschluss (Realschule, Fachoberschule oder Abitur)
- Interesse an IT, Datenanalyse und Geschäftsprozessen
- gute Kenntnisse in Mathematik und logisches Denken
- erste Erfahrungen mit MS Office (insbesondere Excel) von Vorteil
- Teamfähigkeit, Lernbereitschaft und Zuverlässigkeit
- Grundkenntnisse in Englisch (weil viele Tools/Dokumentationen englisch sind)
- wünschenswert: Führerschein der Klasse B (das Unternehmen liegt im Industriegebie

**Bias-Einschätzung**

Mein Korpus umfasst 44 Stellenanzeigen aus der Jobbörse-API der Bundesagentur für Arbeit und ist nicht repräsentativ für alle datenbezogenen IT-Stellen in Deutschland. Die Auswahl ist bereits durch meine Suchanfragen verzerrt, da ich ausschließlich datennahe Begriffe wie Data Analyst, Data Engineer oder Fachinformatiker Daten- und Prozessanalyse verwendet und den Standort Bremen eingeschränkt habe. Trotzdem wurden auch mehrere Stellen außerhalb Bremens zurückgegeben, was auf API-interne Radiussuche, Remote-Stellen oder ungenaue Standortfilter hindeutet. Außerdem enthält der Datensatz fachlich unpassende Treffer wie beispielsweise Electrical Engineer oder Sachbearbeitung Projektabrechnung, obwohl diese nicht dem eigentlichen Suchfokus entsprechen. Das Unternehmen Rheinmetall ist mit mehreren Anzeigen überrepräsentiert, wodurch bestimmte Formulierungen und Anforderungsprofile häufiger vorkommen als andere. Gehaltsangaben fehlen in einem Großteil der Anzeigen, weshalb Aussagen über Vergütung nicht belastbar wären. Zusätzlich unterscheiden sich die Beschreibungstexte stark in Länge, Struktur und Formatierung und enthalten teilweise Markdown-ähnliche Artefakte, was die spätere Information Extraction erschweren kann.

## Phase 2 — κ-Tabelle + drei Edge Cases

In [3]:
fabio_korpus = pd.read_json("../daten/fabio_korpus.jsonl", lines=True)
fabio_korpus.head()

,refnr,titel,firma,text,ort
0,15548-1423375-1-S,"(Senior) Consultant- Financial Crime, Complian...",EY Consulting GmbH,Requisition ID: 1610322\n\n\nAre you ready to ...,"Eschborn, Taunus"
1,10000-1205912617-S,IT-Security Consultant (m/w/d) Schwerpunkt VPN,AirITSystems GmbH,## Du kennst dich bestens mit VPN aus?\n\nDu s...,"Langenhagen, Han"
2,14447-43454028-613-S,Project Manager* Global Procurement,HARTING Stiftung & Co.KG,HARTING steht für starke Verbindungen - rund u...,Espelkamp
3,12942-1555106-1-S,Business Analyst (m/w/d) Service Now mobile Apps,Dirk Rossmann GmbH,Unsere Unternehmenszentrale in Burgwedel bei H...,Burgwedel
4,13509-0000210fdcf001-S,Lead IT Expert DevOps & Platform Engineering (...,BWI GmbH,Sorge gemeinsam mit uns für die digitale Zukun...,Hamburg


In [5]:
meine_ids = set(korpus["refnr"])
partner_ids = set(fabio_korpus["refnr"])

schnittmenge = meine_ids.intersection(partner_ids)

print(f"Meine Anzeigen: {len(meine_ids)}")
print(f"Partner Anzeigen: {len(partner_ids)}")
print(f"Gemeinsame Anzeigen: {len(schnittmenge)}")

Meine Anzeigen: 44
Partner Anzeigen: 163
Gemeinsame Anzeigen: 7


In [9]:
gemeinsame_anzeigen = korpus[
    korpus["refnr"].isin(schnittmenge)]

gemeinsame_anzeigen[["refnr", "titel", "firma", "text", "ort"]]

,refnr,titel,firma,text,ort
5,15939-BB-633097-7878-9999-S,Data Analyst im Marineschiffbau (m/w/d),Rheinmetall AG,Moderne Marineschiffe sind hochkomplexe System...,Bremen
10,15939-BB-633095-7878-7490-S,(Senior) Data Analyst - Lifecycle-Analysen im ...,Rheinmetall AG,Marineschiffe sind hochkomplexe Systeme mit Le...,Bremen
17,15939-BB-633455-7878-6343-S,Data Engineer (m/w/d),Rheinmetall AG,Rheinmetall Digital GmbH\n\nWHAT WE ARE LOOKIN...,Bremen
20,15939-BB-633457-7878-2175-S,MLOps Engineer (m/w/d),Rheinmetall AG,Rheinmetall Digital GmbH\n\nWOFÜR WIR SIE SUCH...,Bremen
26,18777-931781141-S,Requirements Engineer (m/w/d),OHB-System AG,#### Your Tasks\n\n- Being the project’s focal...,Bremen
36,13999-k53401.30280-S,Business Analyst (m/w/d) im Technologiekonzern,jobtimum GmbH Personalvermittlung,\n\nBusiness Analyst (m/w/d) im Technologiekon...,Bremen
42,15939-BB-632493-7878-9058-S,Praktikant Market Intelligence (m/w/d),Rheinmetall AG,* Unterstützung in den Bereichen Market and Co...,Bremen


In [11]:
# pic 5 random entries from fabio_korpus which aren't in gemeinsame anzeigen and create an dataframe with both
# Anzeigen aus Fabios Korpus, die NICHT bereits in der Schnittmenge sind
nur_fabio = fabio_korpus[
    ~fabio_korpus["refnr"].isin(schnittmenge)
]

print(f"Nur Fabio Anzeigen: {len(nur_fabio)}")
zufaellige_fuenf = nur_fabio.sample(
    n=5,
    random_state=42
)

Nur Fabio Anzeigen: 156


In [18]:
annotations_auswahl = pd.concat(
    [gemeinsame_anzeigen, zufaellige_fuenf],
    ignore_index=True
)
annotations_auswahl.head(14)

,refnr,titel,firma,text,ort,beruf,veroeffentlichung,homeoffice_api,gehalt_api,vertragsdauer_api,raw
0,15939-BB-633097-7878-9999-S,Data Analyst im Marineschiffbau (m/w/d),Rheinmetall AG,Moderne Marineschiffe sind hochkomplexe System...,Bremen,Data Scientist,2026-03-18,0.0,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
1,15939-BB-633095-7878-7490-S,(Senior) Data Analyst - Lifecycle-Analysen im ...,Rheinmetall AG,Marineschiffe sind hochkomplexe Systeme mit Le...,Bremen,Data Scientist,2026-03-18,0.0,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
2,15939-BB-633455-7878-6343-S,Data Engineer (m/w/d),Rheinmetall AG,Rheinmetall Digital GmbH\n\nWHAT WE ARE LOOKIN...,Bremen,Fachinformatiker/in - Daten- und Prozessanalyse,2026-04-23,0.0,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
3,15939-BB-633457-7878-2175-S,MLOps Engineer (m/w/d),Rheinmetall AG,Rheinmetall Digital GmbH\n\nWOFÜR WIR SIE SUCH...,Bremen,Machine Learning Engineer,2026-04-23,0.0,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
4,18777-931781141-S,Requirements Engineer (m/w/d),OHB-System AG,#### Your Tasks\n\n- Being the project’s focal...,Bremen,Ingenieur/in - Systems Engineering,2026-03-06,NaN,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
5,13999-k53401.30280-S,Business Analyst (m/w/d) im Technologiekonzern,jobtimum GmbH Personalvermittlung,\n\nBusiness Analyst (m/w/d) im Technologiekon...,Bremen,Business-Analyst/in,2026-04-10,NaN,JAHRESGEHALT,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
6,15939-BB-632493-7878-9058-S,Praktikant Market Intelligence (m/w/d),Rheinmetall AG,* Unterstützung in den Bereichen Market and Co...,Bremen,Data Scientist,2026-04-20,0.0,KEINE_ANGABEN,BEFRISTET,"{'stellenangebotsart': 'PRAKTIKUM_TRAINEE', 's..."
7,13635-7fbe73ac_JB5131141-S,Softwareentwickler:in Machine- / Deep-Learning...,zollsoft GmbH,Wo Du mit anpacken kannst\n • Gleich vom erste...,Hamburg,NaN,NaN,NaN,NaN,NaN,NaN
8,20536-lutif851vc-S,Research Associate / Wissenschaftliche*r Mitar...,Technische Universität Hamburg,Research Associate (m/f/d) - \nWissenschaftl...,Hamburg,NaN,NaN,NaN,NaN,NaN,NaN
9,12826-SA0136034_JB5125696-S,Fachinformatiker für Systemintegration (m/w/d),Piening GmbH Engineering & IT,Wir suchen Dich!\nPiening gehört zu den Top En...,Hamburg,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# save annotations_auswahl to csv

annotations_auswahl.to_csv("../daten/annotaions_auswahl.csv", index=False)